In [6]:
import sys
sys.path.append('../src')

In [7]:
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sns 
import numpy as np 

In [8]:
dataset = pd.read_csv("AmesHousing.csv")

In [9]:
from feature_engineering import add_engineered_features

dataset = add_engineered_features(dataset)

In [10]:
from preprocessing import get_preprocessor  
from feature_target import add_log_target

preprocessor = get_preprocessor() 

dataset = add_log_target(dataset, target_col='SalePrice')

# Ridge Regression

In [11]:
from sklearn.linear_model import Ridge 
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ('preprocessor',preprocessor),
    ('model',Ridge(alpha=1.0))
])

## splitting

In [12]:
y = dataset['SalePrice_log']
X = dataset.drop(columns=['SalePrice','SalePrice_log'])

## As Central Air is object type so i got error at the time of fit(X_train, y_train), so that's why i map this feture to numeric (0,1)

In [13]:
X['Central Air'] = X['Central Air'].map({'Y': 1, 'N': 0})

### Train test split

In [14]:
from sklearn.model_selection import train_test_split 

X_train , X_valid , y_train , y_valid = train_test_split(X,y,test_size =0.2,random_state =42)
pipeline.fit(X_train,y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['House Age',
                                                   'Year since Remodel',
                                                   'Total Living Area',
                                                   'Total Finished Area',
                                                   'Total Bathrooms',
                                                   'Garage Age',
                                                   'Quality_size_score',
                                                   'Non_Bedroom_Rooms']),
                                                 ('ord',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer...
                                                 ('nom_high',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['Neighborhood',
                                                   'Exterior 1st',
                                                   'Exterior 2nd']),
                                                 ('bin',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent'))]),
                                                  ['Central Air',
                                                   'Has_Basement',
                                                   'Is_Remodeled',
                                                   'Garage Exists'])])),
                ('model', Ridge())])

In [15]:
y_pred = pipeline.predict(X_valid)

##  Calculating RMSE value after trained data using Ridge Regression model (target feature is log(SalePrice)

In [16]:
from sklearn.metrics import root_mean_squared_error

rmse = root_mean_squared_error(y_valid, y_pred)
rmse

0.1326434996287248

Model Evaluation Note
The RMSE value of 0.137 is computed on the log-transformed SalePrice, not on the original price scale. This corresponds to an average prediction error of approximately 14–15% in actual house prices, which is a good result for a baseline Ridge Regression model

Note : 0.13704642209675533 is log_error we can convert it into percentage error using formula : => percentage error ≈ (e^RMSE − 1) × 100

## So , Now I stop focusing on accuracy and start focusing on explanation.
### As Ridge learned weights for every transformed feature and these weights tell us which factors matter more

In [17]:
coef = pipeline.named_steps['model'].coef_

##  Get Features name 

In [18]:
feature_names = pipeline.named_steps['preprocessor'].get_feature_names_out()
import pandas as pd

coef_df = pd.DataFrame({
    'feature': feature_names,
    'coefficient': coef
}).sort_values(by='coefficient', ascending=False)

So coef_df tells us:
“If ONLY this one feature changes, and everything else stays the same, how much will the model increase or decrease the predicted house price?”
for eg : below you see ((nom_low__Misc Feature_Shed)) has highest coef values which means if a house has a shed then model predicts the log(price) will increase by 0.31 assuming all other features are unchanged and this also make sense house has shed is more precious

In [19]:
coef_df.head()

,feature,coefficient
154,nom_high__Neighborhood_GrnHill,0.308024
118,nom_low__Misc Feature_Shed,0.305727
117,nom_low__Misc Feature_Othr,0.301526
116,nom_low__Misc Feature_Gar2,0.264087
92,nom_low__Roof Matl_WdShngl,0.223720


Note : “House quality and usable space have the strongest positive impact on price, while older construction and poor basement quality reduce value. Renovation has a clear positive effect even for older houses.”

Note: House prices do not have a purely linear relationship with all features; many variables exhibit non-linear effects and complex interactions. Therefore, instead of relying only on Ridge Regression (a linear model), we also train a Random Forest model, which can naturally capture non-linear patterns and feature interactions

# Random Forest 

In [33]:
from sklearn.ensemble import RandomForestRegressor
rf_pipeline = Pipeline([
    ('preprocessor',preprocessor),
    ('model',RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ))
])
rf_pipeline.fit(X_train, y_train)

rf_pred = rf_pipeline.predict(X_valid)


In [34]:
from sklearn.metrics import root_mean_squared_error
rf_rmse = root_mean_squared_error(y_valid, rf_pred)

rf_rmse

0.12072912998951109

## Random Forest achieved a lower RMSE (0.120) compared to Ridge (0.132), indicating better performance by modeling non-linear relationships, while Ridge served as a strong linear baseline.

The performance gap between Ridge and Random Forest is moderate, which indicates that the feature engineering is strong and the dataset contains both linear and non-linear relationships. Ridge was used as a baseline model to validate feature quality, while Random Forest demonstrates the benefit of modeling non-linearity.


## Feature Importance from Random Forest

In [36]:
rf_importance = rf_pipeline.named_steps['model'].feature_importances_
feature_names = rf_pipeline.named_steps['preprocessor'].get_feature_names_out()

rf_imp_df = (
    pd.DataFrame({
        'feature': feature_names,
        'importance': rf_importance
    })
    .sort_values(by='importance', ascending=False)
)


In [37]:
rf_imp_df.head(10)

,feature,importance
6,num__Quality_size_score,0.693882
2,num__Total Living Area,0.049135
3,num__Total Finished Area,0.041745
0,num__House Age,0.034374
10,ord__Bsmt Qual,0.019599
1,num__Year since Remodel,0.015801
13,ord__Kitchen Qual,0.011441
8,ord__Exter Qual,0.009089
5,num__Garage Age,0.007703
204,bin__Central Air,0.006892


#### AS we observe in Random Forest ,Quality_size_score is the most important feature because it combines living area and construction quality, two primary drivers of house prices, allowing the Random Forest model to capture their joint impact more effectively

# Now it's time to SHAP (SHapley Additive exPlanations) , Now Question is what is SHAP , so to understand SHAP I write down the definition of SHAP

What is SHAP?  
SHAP is a method that helps us understand how a machine learning model makes decisions. It tells us how much each input (feature) is helping or hurting the final prediction. The main idea is to fairly distribute the "payout" (the prediction) among all features based on their contribution